# Theta phase precession from button presses

This notebook samples the 2–5 Hz theta phase at each UPPT002 button press, calculates the linear-circular correlation coefficient for press order and phase within each trial, averages trial values within each participant, and plots the participant-weighted group mean for each MEG sensor.

Button press times in UPPT002_button_press_times.npz are in seconds relative to cue onset and are stored as participant × trial × press slot. The subject_ID array gives the participant row order. Subject 11 is excluded before analysis. Subject 16 has 222 trials; the other participants are expected to have 240.

The circular-linear coefficient used here is the standard unsigned coefficient based on the multiple correlation of press order with sin(phase) and cos(phase). Values range from 0 to 1, so this coefficient measures association strength but does not indicate whether phase advances or precesses. Each trial has at most five presses, making trial-level estimates noisy and positively biased; interpret the scalp map descriptively. A directional phase-precession claim would also need a signed circular-linear slope and an appropriate null or surrogate analysis.

import packages for loading MEG epochs, filtering signals and plotting results

In [ ]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

define the theta band, file locations, excluded participant and channel batch size

In [ ]:
# Use the repository network share by default; set MEG_DATA_ROOT for a local export.
DATA_ROOT = os.environ.get(
    "MEG_DATA_ROOT",
    r"//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
)

BUTTON_PRESS_TIMES_FILE = os.path.join(
    os.getcwd(), "UPPT002_button_press_times.npz"
)

theta_freq_band = np.array([2.0, 5.0])  # theta band edges in Hz
excluded_subjects = ["11"]              # participants excluded from the analysis
CHANNEL_BATCH_SIZE = 8                  # channels filtered at a time to limit memory use

# Require the full button-press sequence so trial estimates use the same press positions.
MIN_PRESSES_PER_TRIAL = 5

load and check UPPT002 button-press times

In [ ]:
def normalise_subject_id(id):
    # Keep participant IDs in the repository's two-digit string format.
    return str(int(str(id).strip().lower().lstrip("s"))).zfill(2)


if not os.path.isfile(BUTTON_PRESS_TIMES_FILE):
    raise FileNotFoundError(
        "Could not find UPPT002_button_press_times.npz at "
        + BUTTON_PRESS_TIMES_FILE
    )

with np.load(BUTTON_PRESS_TIMES_FILE, allow_pickle=False) as press_file:
    required_keys = [
        "uppt002_times",
        "subject_ID",
        "n_trials_by_subject",
        "press_slot",
        "cue_onset_after_epoch_start_s",
        "time_reference",
    ]
    missing_keys = [key for key in required_keys if key not in press_file]
    if missing_keys:
        raise KeyError("Missing keys in button-press file: " + str(missing_keys))

    UPPT002_times = np.asarray(press_file["uppt002_times"], dtype=float)
    subject_ID = np.asarray(press_file["subject_ID"]).astype(str)
    n_trials_by_subject = np.asarray(
        press_file["n_trials_by_subject"], dtype=int
    )
    press_slot = np.asarray(press_file["press_slot"], dtype=int)
    CUE_ONSET_AFTER_EPOCH_START_S = float(
        np.asarray(press_file["cue_onset_after_epoch_start_s"]).item()
    )
    time_reference = np.asarray(press_file["time_reference"]).item()

subject_ID = np.asarray(
    [normalise_subject_id(id) for id in subject_ID], dtype=str
)

if isinstance(time_reference, bytes):
    time_reference = time_reference.decode("utf-8")

if "relative to cue onset" not in str(time_reference).lower():
    raise ValueError(
        "Expected button-press times in seconds relative to cue onset; got "
        + str(time_reference)
    )

if UPPT002_times.ndim != 3:
    raise ValueError(
        "Expected uppt002_times in participant × trial × press-slot order, got "
        + str(UPPT002_times.shape)
    )
if len(subject_ID) != UPPT002_times.shape[0]:
    raise ValueError("subject_ID does not match the participant dimension")
if len(np.unique(subject_ID)) != len(subject_ID):
    raise ValueError("subject_ID contains duplicate participant IDs")
if n_trials_by_subject.shape != (len(subject_ID),):
    raise ValueError("n_trials_by_subject does not match subject_ID")
if np.any(n_trials_by_subject < 1) or np.any(
    n_trials_by_subject > UPPT002_times.shape[1]
):
    raise ValueError("n_trials_by_subject contains an invalid trial count")
if press_slot.shape != (UPPT002_times.shape[2],):
    raise ValueError("press_slot does not match the press-slot dimension")
if not np.array_equal(press_slot, np.arange(1, len(press_slot) + 1)):
    raise ValueError("Expected press slots numbered in sequence from 1")

n_press_slots = len(press_slot)
if MIN_PRESSES_PER_TRIAL > n_press_slots:
    raise ValueError("MIN_PRESSES_PER_TRIAL exceeds the number of press slots")

# Use the subject order stored in the NPZ file and remove excluded subjects.
sub_ID = [id for id in subject_ID if id not in excluded_subjects]
if len(sub_ID) == 0:
    raise ValueError("No participants remain after applying excluded_subjects.")

subject_index = {id: index for index, id in enumerate(subject_ID)}

print("Button-press array shape:", UPPT002_times.shape)
print("Participants to process:", sub_ID)
print("Excluded participants:", excluded_subjects)
print("Press slots:", press_slot.tolist())
print("Time reference:", time_reference)

load one participant's MEG epochs and identify the MEG sensors

In [ ]:
def filename_loader(id):
    # Build the participant's MEG epoch path using the repository naming pattern.
    subject_number = str(int(id))
    subject_folder = "s" + str(id)
    file_name = "Subject" + subject_number + "_spm_mne-epo.fif"

    epochs_file = os.path.join(DATA_ROOT, subject_folder, file_name)
    if not os.path.isfile(epochs_file):
        # Also support a local export with the FIF file at the data-root level.
        epochs_file = os.path.join(DATA_ROOT, file_name)

    if not os.path.isfile(epochs_file):
        raise FileNotFoundError(
            "Could not find " + file_name + " under " + DATA_ROOT
        )

    # Read epoch metadata first; sensor batches are loaded during analysis.
    epochs = mne.read_epochs(epochs_file, preload=False, verbose="ERROR")

    # MEG sensors use names beginning with M in this repository.
    MEG_sens = [
        channel for channel in epochs.ch_names if channel.startswith("M")
    ]
    if len(MEG_sens) == 0:
        raise ValueError("No MEG sensors were found in " + epochs_file)

    return epochs, MEG_sens

calculate the circular-linear correlation coefficient for phase and press order

In [ ]:
def linear_circular_corrcoef(phase_by_sensor, linear_values, min_n=5):
    # Calculate Mardia's circular-linear coefficient for each sensor.
    # phase_by_sensor has shape sensors × presses; linear_values are press slots.
    phase_by_sensor = np.asarray(phase_by_sensor, dtype=float)
    linear_values = np.asarray(linear_values, dtype=float)

    if phase_by_sensor.ndim != 2:
        raise ValueError("phase_by_sensor must be a sensors-by-presses array")
    if linear_values.ndim != 1 or phase_by_sensor.shape[1] != len(linear_values):
        raise ValueError("Press order and phase arrays have incompatible shapes")

    n_sensors = phase_by_sensor.shape[0]
    coefficient = np.full(n_sensors, np.nan)

    for sensor_index in range(n_sensors):
        phase = phase_by_sensor[sensor_index]
        valid = np.isfinite(phase) & np.isfinite(linear_values)
        if valid.sum() < min_n:
            continue

        x = linear_values[valid]
        angle = phase[valid]
        x_centered = x - x.mean()
        sine = np.sin(angle)
        cosine = np.cos(angle)
        sine_centered = sine - sine.mean()
        cosine_centered = cosine - cosine.mean()

        sum_x2 = np.sum(x_centered ** 2)
        sum_sine2 = np.sum(sine_centered ** 2)
        sum_cosine2 = np.sum(cosine_centered ** 2)
        if min(sum_x2, sum_sine2, sum_cosine2) <= np.finfo(float).eps:
            continue

        r_x_sine = np.sum(x_centered * sine_centered) / np.sqrt(
            sum_x2 * sum_sine2
        )
        r_x_cosine = np.sum(x_centered * cosine_centered) / np.sqrt(
            sum_x2 * sum_cosine2
        )
        r_sine_cosine = np.sum(sine_centered * cosine_centered) / np.sqrt(
            sum_sine2 * sum_cosine2
        )

        denominator = 1.0 - r_sine_cosine ** 2
        if denominator <= np.finfo(float).eps:
            continue

        r_squared = (
            r_x_sine ** 2
            + r_x_cosine ** 2
            - 2.0 * r_x_sine * r_x_cosine * r_sine_cosine
        ) / denominator

        if np.isfinite(r_squared):
            # The coefficient is unsigned and lies between zero and one.
            coefficient[sensor_index] = np.sqrt(np.clip(r_squared, 0.0, 1.0))

    return coefficient

filter each MEG sensor in the 2–5 Hz theta band, extract the Hilbert phase at each button press, then average trial coefficients within the participant

In [ ]:
def participant_phase_precession(id):
    # Load the participant's epochs and the matching UPPT002 timing rows.
    epochs, MEG_sens = filename_loader(id)
    data_index = subject_index[id]
    n_trials = int(n_trials_by_subject[data_index])

    if len(epochs) != n_trials:
        raise ValueError(
            "Participant " + id + " has " + str(len(epochs))
            + " MEG epochs but " + str(n_trials) + " saved button-press trials"
        )

    press_times = UPPT002_times[data_index, :n_trials, :]
    if press_times.shape[1] != n_press_slots:
        raise ValueError("Unexpected number of button-press slots for " + id)

    # Map cue-relative times onto the epoch time axis.
    cue_onset_epoch_s = epochs.times[0] + CUE_ONSET_AFTER_EPOCH_START_S
    event_times = press_times + cue_onset_epoch_s
    sfreq = float(epochs.info["sfreq"])

    # Convert each event time to its nearest epoch sample.
    sample_indices = np.full(press_times.shape, -1, dtype=int)
    finite_press = np.isfinite(event_times)
    sample_indices[finite_press] = np.rint(
        (event_times[finite_press] - epochs.times[0]) * sfreq
    ).astype(int)
    valid_press = (
        finite_press
        & (sample_indices >= 0)
        & (sample_indices < len(epochs.times))
    )
    complete_trial = (
        valid_press.sum(axis=1) >= MIN_PRESSES_PER_TRIAL
    )

    # Save phase at each usable press in trials × sensors × press-slots order.
    phase_by_trial = np.full(
        (n_trials, len(MEG_sens), n_press_slots), np.nan, dtype=float
    )

    for batch_start in range(0, len(MEG_sens), CHANNEL_BATCH_SIZE):
        batch_stop = min(batch_start + CHANNEL_BATCH_SIZE, len(MEG_sens))
        channel_batch = MEG_sens[batch_start:batch_stop]

        # Load a small channel batch, filter it to theta, and obtain phase.
        epochs_batch = epochs.copy().pick(channel_batch).load_data()
        epochs_batch.filter(
            l_freq=float(theta_freq_band[0]),
            h_freq=float(theta_freq_band[1]),
            picks=channel_batch,
            verbose="ERROR",
        )
        epochs_batch.apply_hilbert(
            picks=channel_batch,
            envelope=False,
            verbose="ERROR",
        )
        analytic_data = epochs_batch.get_data(picks=channel_batch)

        if analytic_data.shape[0] != n_trials:
            raise ValueError(
                "Filtering changed the trial count for participant " + id
            )

        for trial_index in np.flatnonzero(complete_trial):
            for press_index in np.flatnonzero(valid_press[trial_index]):
                sample_index = sample_indices[trial_index, press_index]
                phase_by_trial[
                    trial_index, batch_start:batch_stop, press_index
                ] = np.angle(analytic_data[trial_index, :, sample_index])

        del analytic_data
        del epochs_batch

    # Calculate one coefficient per trial and MEG sensor.
    trial_lncc = np.full((n_trials, len(MEG_sens)), np.nan, dtype=float)
    trial_rows = []

    for trial_index in range(n_trials):
        valid_slots = np.flatnonzero(
            np.isfinite(phase_by_trial[trial_index]).any(axis=0)
        )
        n_valid_presses = int(valid_press[trial_index].sum())

        if len(valid_slots) >= MIN_PRESSES_PER_TRIAL:
            trial_lncc[trial_index] = linear_circular_corrcoef(
                phase_by_trial[trial_index][:, valid_slots],
                press_slot[valid_slots],
                min_n=MIN_PRESSES_PER_TRIAL,
            )

        trial_rows.append({
            "participant": id,
            "trial": trial_index + 1,
            "n_valid_presses": n_valid_presses,
            "complete_press_sequence": (
                n_valid_presses >= MIN_PRESSES_PER_TRIAL
            ),
            "n_sensors_with_lncc": int(
                np.isfinite(trial_lncc[trial_index]).sum()
            ),
        })

    # Average trial-level coefficients within this participant for each sensor.
    trial_counts = np.isfinite(trial_lncc).sum(axis=0)
    participant_mean = np.divide(
        np.nansum(trial_lncc, axis=0),
        trial_counts,
        out=np.full(len(MEG_sens), np.nan),
        where=trial_counts > 0,
    )

    # Keep sensor geometry for the later scalp map.
    meg_picks = mne.pick_channels(epochs.ch_names, include=MEG_sens)
    sensor_info = mne.pick_info(epochs.info, meg_picks, copy=True)

    participant_summary = {
        "participant": id,
        "n_trials": n_trials,
        "n_trials_with_lncc": int(np.isfinite(trial_lncc).any(axis=1).sum()),
        "n_complete_press_trials": int(complete_trial.sum()),
        "n_valid_press_times": int(valid_press.sum()),
        "mean_lncc_across_sensors": (
            float(np.nanmean(participant_mean))
            if np.isfinite(participant_mean).any() else np.nan
        ),
    }

    return (
        participant_mean,
        MEG_sens,
        sensor_info,
        participant_summary,
        trial_rows,
    )

process included participants separately; subject 11 is excluded before this loop

In [ ]:
participant_lncc = []
participant_rows = []
trial_rows = []
group_sensors = None
group_info = None

for id in sub_ID:
    (
        participant_mean,
        MEG_sens,
        sensor_info,
        participant_summary,
        subject_trial_rows,
    ) = participant_phase_precession(id)

    # Confirm that channel values refer to the same sensors for every participant.
    if group_sensors is None:
        group_sensors = MEG_sens
        group_info = sensor_info
    elif MEG_sens != group_sensors:
        raise ValueError(
            "MEG sensor names or order differ for participant " + id
        )

    participant_lncc.append(participant_mean)
    participant_rows.append(participant_summary)
    trial_rows.extend(subject_trial_rows)

    print(
        "Participant", id,
        "- complete trials:", participant_summary["n_complete_press_trials"],
        "- trials with LNCC:", participant_summary["n_trials_with_lncc"],
    )

participant_lncc = np.asarray(participant_lncc, dtype=float)
participant_summary_table = pd.DataFrame(participant_rows)
trial_qc_table = pd.DataFrame(trial_rows)

print("Participant-by-sensor LNCC shape:", participant_lncc.shape)
print("Sensors:", len(group_sensors))

review participant and trial values, then calculate an equal-weight mean across participant means

In [ ]:
# Report participant-level values and trial-level timing quality.
subject_lncc_table = pd.DataFrame(
    participant_lncc,
    index=sub_ID,
    columns=group_sensors,
)
subject_lncc_table.index.name = "participant"

n_participants_by_sensor = np.isfinite(participant_lncc).sum(axis=0)
group_lncc = np.divide(
    np.nansum(participant_lncc, axis=0),
    n_participants_by_sensor,
    out=np.full(len(group_sensors), np.nan),
    where=n_participants_by_sensor > 0,
)

group_lncc_table = pd.DataFrame({
    "sensor": group_sensors,
    "mean_lncc": group_lncc,
    "n_participants": n_participants_by_sensor,
})

print("Participant summary:")
display(participant_summary_table.round(4))

print("Per-participant mean LNCC by sensor:")
display(subject_lncc_table.round(4))

print("Group mean across participant means:")
display(group_lncc_table.round(4))

incomplete_trials = trial_qc_table[
    ~trial_qc_table["complete_press_sequence"]
]
print("Trials with fewer than", MIN_PRESSES_PER_TRIAL, "valid presses:")
display(incomplete_trials.head(20))

plot a scalp map of the group mean for each MEG sensor type

In [ ]:
# Plot the participant-weighted group mean for each MEG sensor type.
sensor_types = [
    mne.channel_type(group_info, index)
    for index in range(len(group_sensors))
]

for sensor_type in sorted(set(sensor_types)):
    picks = np.array([
        index for index, kind in enumerate(sensor_types)
        if kind == sensor_type
    ])
    sensor_info = mne.pick_info(group_info, picks, copy=True)

    fig, ax = plt.subplots(figsize=(7, 6))
    image, _ = mne.viz.plot_topomap(
        group_lncc[picks],
        sensor_info,
        ch_type=sensor_type,
        axes=ax,
        cmap="viridis",
        contours=0,
        show=False,
    )
    image.set_clim(0.0, 1.0)
    ax.set_title(
        "Mean participant 2–5 Hz theta phase-precession LNCC ("
        + sensor_type + ")"
    )
    fig.colorbar(image, ax=ax, label="Linear-circular correlation coefficient")
    plt.tight_layout()
    plt.show()